In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

root_dir = Path('/home/aparnabg/orcd/scratch/ml_models')
data_path = root_dir / 'data.csv'
preprocessing_dir = root_dir / 'data_preprocessing'
preprocessing_dir.mkdir(exist_ok=True)

df = pd.read_csv(data_path)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded: 3411 rows, 46 columns


In [2]:
print("Original columns:")
print(df.columns.tolist())
print(f"\nTarget variable distribution:")
print(df['Group'].value_counts())

Original columns:
['Coder', 'SourceFile', 'ID', 'FileName', 'Vid_duration', 'DOB', 'VideoDate', 'Age', 'timepoint', 'DateValidityScore', 'Context', 'Location', 'Activity', 'Child_of_interest_clear', '#_adults', '#_children', '#_people_background', 'Interaction_with_child', '#_people_interacting', 'Child_constrained', 'Constraint_type', 'Supports', 'Support_type', 'Example_support_type', 'Gestures', 'Gesture_type', 'Vocalizations', 'RMM', 'RMM_type', 'Response_to_name', 'Locomotion', 'Locomotion_type', 'Grasping', 'Grasp_type', 'Body_Parts_Visible', 'Angle_of_Body', 'Video_Quality_Child_Face_Visibility', 'Video_Quality_Child_Body_Visibility', 'Video_Quality_Child_Hand_Visibility', 'Video_Quality_Lighting', 'Video_Quality_Resolution', 'Video_Quality_Motion', 'Notes', 'BidsProcessed', 'BidsRaw', 'Group']

Target variable distribution:
Group
ASD        2309
Non-ASD    1102
Name: count, dtype: int64


In [3]:
columns_to_drop = {
    'Identifiers': [
        'Coder',
        'SourceFile',
        'ID',
        'FileName',
        'DOB',
        'VideoDate',
        'BidsProcessed',
        'BidsRaw',
    ],
    
    'Recording Quality': [
        'Vid_duration',
        'DateValidityScore',
        'Child_of_interest_clear',
        'Body_Parts_Visible',
        'Angle_of_Body',
        'Video_Quality_Child_Face_Visibility',
        'Video_Quality_Child_Body_Visibility',
        'Video_Quality_Child_Hand_Visibility',
        'Video_Quality_Lighting',
        'Video_Quality_Resolution',
        'Video_Quality_Motion',
    ],
    
    'High Cardinality': [
        'Activity',
        'Example_support_type',
        'Constraint_type',
    ],
    
    'Other': [
        'Notes',
    ]
}

for category, cols in columns_to_drop.items():
    print(f"\n{category}:")
    for col in cols:
        print(f"  - {col}")

all_cols_to_drop = [col for cols in columns_to_drop.values() for col in cols]
print(f"\nDropping {len(all_cols_to_drop)} columns total")


Identifiers:
  - Coder
  - SourceFile
  - ID
  - FileName
  - DOB
  - VideoDate
  - BidsProcessed
  - BidsRaw

Recording Quality:
  - Vid_duration
  - DateValidityScore
  - Child_of_interest_clear
  - Body_Parts_Visible
  - Angle_of_Body
  - Video_Quality_Child_Face_Visibility
  - Video_Quality_Child_Body_Visibility
  - Video_Quality_Child_Hand_Visibility
  - Video_Quality_Lighting
  - Video_Quality_Resolution
  - Video_Quality_Motion

High Cardinality:
  - Activity
  - Example_support_type
  - Constraint_type

Other:
  - Notes

Dropping 23 columns total


In [4]:
df_clean = df.drop(columns=all_cols_to_drop)
print(f"After dropping: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
print(f"\nRemaining columns:")
print(df_clean.columns.tolist())

After dropping: 3411 rows, 23 columns

Remaining columns:
['Age', 'timepoint', 'Context', 'Location', '#_adults', '#_children', '#_people_background', 'Interaction_with_child', '#_people_interacting', 'Child_constrained', 'Supports', 'Support_type', 'Gestures', 'Gesture_type', 'Vocalizations', 'RMM', 'RMM_type', 'Response_to_name', 'Locomotion', 'Locomotion_type', 'Grasping', 'Grasp_type', 'Group']


In [5]:
print("Data types:")
print(df_clean.dtypes)

Data types:
Age                       float64
timepoint                  object
Context                    object
Location                   object
#_adults                   object
#_children                 object
#_people_background        object
Interaction_with_child     object
#_people_interacting       object
Child_constrained          object
Supports                   object
Support_type               object
Gestures                   object
Gesture_type               object
Vocalizations              object
RMM                        object
RMM_type                   object
Response_to_name           object
Locomotion                 object
Locomotion_type            object
Grasping                   object
Grasp_type                 object
Group                      object
dtype: object


In [6]:
print("Missing values:")
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean)) * 100
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing': missing.values,
    'Percent': missing_pct.values
}).sort_values('Missing', ascending=False)
print(missing_df[missing_df['Missing'] > 0])

Missing values:
                    Column  Missing    Percent
17        Response_to_name     3175  93.081208
16                RMM_type     3037  89.035473
19         Locomotion_type     1980  58.047493
13            Gesture_type     1796  52.653181
21              Grasp_type     1484  43.506303
11            Support_type     1105  32.395192
1                timepoint      994  29.141014
0                      Age      253   7.417180
18              Locomotion       78   2.286719
12                Gestures       76   2.228086
9        Child_constrained       75   2.198769
6      #_people_background       75   2.198769
15                     RMM       75   2.198769
20                Grasping       75   2.198769
14           Vocalizations       75   2.198769
5               #_children       74   2.169452
4                 #_adults       74   2.169452
3                 Location       74   2.169452
2                  Context       74   2.169452
8     #_people_interacting       74   2.1694

In [7]:
print("Numeric columns summary:")
print(df_clean.select_dtypes(include=[np.number]).describe())

Numeric columns summary:
               Age
count  3158.000000
mean      2.073767
std       0.940922
min       0.826800
25%       1.169100
50%       1.542750
75%       3.000700
max       4.692700


In [8]:
print("Categorical columns:")
cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
if 'Group' in cat_cols:
    cat_cols.remove('Group')
    
for col in cat_cols:
    unique_count = df_clean[col].nunique()
    print(f"\n{col}: {unique_count} unique values")
    print(df_clean[col].value_counts())

Categorical columns:

timepoint: 2 unique values
timepoint
14_month    1274
36_month    1143
Name: count, dtype: int64

Context: 9 unique values
Context
general social communication interaction    1101
toy play                                     658
motor play                                   566
other                                        326
daily routine                                285
social routine                               177
special occasion                             149
book share                                    74
general social interaction                     1
Name: count, dtype: int64

Location: 6 unique values
Location
inside private     2463
outside public      339
inside public       279
outside private     244
unknown              11
outside oublic        1
Name: count, dtype: int64

#_adults: 8 unique values
#_adults
0      2486
1       750
2        68
3        22
4         5
10+       3
7         2
5         1
Name: count, dtype: int64

#_children: 11 

In [9]:
print("First few rows:")
print(df_clean.head())

First few rows:
      Age timepoint                                   Context        Location  \
0  0.9528       NaN                          special occasion  inside private   
1  0.9528       NaN  general social communication interaction  inside private   
2  0.9665       NaN                          special occasion  inside private   
3  0.9938       NaN                                motor play  inside private   
4  1.0048  14_month  general social communication interaction  inside private   

  #_adults #_children #_people_background Interaction_with_child  \
0        1          2                   1                    yes   
1        0          1                   1                    yes   
2        1          1                   0                    yes   
3        0          1                   0                    yes   
4        1          1                   0                    yes   

  #_people_interacting Child_constrained  ... Gesture_type Vocalizations RMM  \
0       

In [10]:
print("Current state:")
print(f"Shape: {df_clean.shape}")
print(f"\nColumns with missing data:")
for col in df_clean.columns:
    missing = df_clean[col].isnull().sum()
    if missing > 0:
        pct = (missing / len(df_clean)) * 100
        print(f"{col}: {missing} ({pct:.1f}%)")

Current state:
Shape: (3411, 23)

Columns with missing data:
Age: 253 (7.4%)
timepoint: 994 (29.1%)
Context: 74 (2.2%)
Location: 74 (2.2%)
#_adults: 74 (2.2%)
#_children: 74 (2.2%)
#_people_background: 75 (2.2%)
Interaction_with_child: 74 (2.2%)
#_people_interacting: 74 (2.2%)
Child_constrained: 75 (2.2%)
Supports: 74 (2.2%)
Support_type: 1105 (32.4%)
Gestures: 76 (2.2%)
Gesture_type: 1796 (52.7%)
Vocalizations: 75 (2.2%)
RMM: 75 (2.2%)
RMM_type: 3037 (89.0%)
Response_to_name: 3175 (93.1%)
Locomotion: 78 (2.3%)
Locomotion_type: 1980 (58.0%)
Grasping: 75 (2.2%)
Grasp_type: 1484 (43.5%)


In [11]:
df_clean['Location'] = df_clean['Location'].replace('outside oublic', 'outside public')
df_clean['Gesture_type'] = df_clean['Gesture_type'].replace({
    'clapping': 'clap',
    'mutiple': 'multiple'
})
print("Fixed typos in Location and Gesture_type")

Fixed typos in Location and Gesture_type


In [12]:
def clean_count_column(col):
    col = col.astype(str)
    col = col.replace('+', '10')
    col = col.replace('5+', '5')
    col = col.replace('10+', '10')
    return pd.to_numeric(col, errors='coerce')

df_clean['#_adults'] = clean_count_column(df_clean['#_adults'])
df_clean['#_children'] = clean_count_column(df_clean['#_children'])
df_clean['#_people_background'] = clean_count_column(df_clean['#_people_background'])
df_clean['#_people_interacting'] = clean_count_column(df_clean['#_people_interacting'])

print("Converted count columns to numeric:")
print(df_clean[['#_adults', '#_children', '#_people_background', '#_people_interacting']].describe())

Converted count columns to numeric:
          #_adults   #_children  #_people_background  #_people_interacting
count  3337.000000  3337.000000          3336.000000           3337.000000
mean      0.305963     1.277495             0.341727              1.010788
std       0.650265     0.766708             1.291238              0.966404
min       0.000000     0.000000             0.000000              0.000000
25%       0.000000     1.000000             0.000000              1.000000
50%       0.000000     1.000000             0.000000              1.000000
75%       1.000000     1.000000             0.000000              1.000000
max      10.000000    10.000000            10.000000             10.000000


In [13]:
print("\nColumns with high missing values:")
high_missing = ['Response_to_name', 'RMM_type', 'Grasp_type', 
                'Locomotion_type', 'Gesture_type', 'Support_type']

for col in high_missing:
    missing_pct = (df_clean[col].isnull().sum() / len(df_clean)) * 100
    print(f"{col}: {missing_pct:.1f}% missing")


Columns with high missing values:
Response_to_name: 93.1% missing
RMM_type: 89.0% missing
Grasp_type: 43.5% missing
Locomotion_type: 58.0% missing
Gesture_type: 52.7% missing
Support_type: 32.4% missing


In [14]:
print("\nWhat about timepoint? 29% missing")
print(df_clean['timepoint'].value_counts(dropna=False))
print("\nAge distribution when timepoint is missing:")
print(df_clean[df_clean['timepoint'].isnull()]['Age'].describe())
print("\nAge distribution for 14_month:")
print(df_clean[df_clean['timepoint'] == '14_month']['Age'].describe())
print("\nAge distribution for 36_month:")
print(df_clean[df_clean['timepoint'] == '36_month']['Age'].describe())


What about timepoint? 29% missing
timepoint
14_month    1274
36_month    1143
NaN          994
Name: count, dtype: int64

Age distribution when timepoint is missing:
count    741.000000
mean       2.110609
std        0.990797
min        0.826800
25%        0.999300
50%        1.943900
75%        2.817200
max        4.692700
Name: Age, dtype: float64

Age distribution for 14_month:
count    1274.000000
mean        1.193671
std         0.121311
min         1.002100
25%         1.086900
50%         1.185500
75%         1.303200
max         1.415500
Name: Age, dtype: float64

Age distribution for 36_month:
count    1143.000000
mean        3.030847
std         0.114713
min         2.833700
25%         2.940500
50%         3.019800
75%         3.122500
max         3.249800
Name: Age, dtype: float64


In [15]:
print(f"\nBefore dropping Age missing: {df_clean.shape}")
df_clean = df_clean[df_clean['Age'].notna()]
print(f"After dropping Age missing: {df_clean.shape}")
print(f"Dropped {3411 - len(df_clean)} rows")


Before dropping Age missing: (3411, 23)
After dropping Age missing: (3158, 23)
Dropped 253 rows


In [16]:
print("\nFilling remaining missing values with 'none':")
for col in df_clean.columns:
    if col != 'Group' and df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype == 'object':
            df_clean[col] = df_clean[col].fillna('none')
            print(f"{col}: filled with 'none'")
        else:
            df_clean[col] = df_clean[col].fillna(0)
            print(f"{col}: filled with 0")


Filling remaining missing values with 'none':
timepoint: filled with 'none'
Context: filled with 'none'
Location: filled with 'none'
#_adults: filled with 0
#_children: filled with 0
#_people_background: filled with 0
Interaction_with_child: filled with 'none'
#_people_interacting: filled with 0
Child_constrained: filled with 'none'
Supports: filled with 'none'
Support_type: filled with 'none'
Gestures: filled with 'none'
Gesture_type: filled with 'none'
Vocalizations: filled with 'none'
RMM: filled with 'none'
RMM_type: filled with 'none'
Response_to_name: filled with 'none'
Locomotion: filled with 'none'
Locomotion_type: filled with 'none'
Grasping: filled with 'none'
Grasp_type: filled with 'none'


In [17]:
print("\nChecking for any remaining missing:")
print(df_clean.isnull().sum().sum())
print("\nFinal shape:")
print(df_clean.shape)


Checking for any remaining missing:
0

Final shape:
(3158, 23)


In [18]:
print("\nFinal columns:")
print(df_clean.columns.tolist())
print(f"\nTarget distribution:")
print(df_clean['Group'].value_counts())


Final columns:
['Age', 'timepoint', 'Context', 'Location', '#_adults', '#_children', '#_people_background', 'Interaction_with_child', '#_people_interacting', 'Child_constrained', 'Supports', 'Support_type', 'Gestures', 'Gesture_type', 'Vocalizations', 'RMM', 'RMM_type', 'Response_to_name', 'Locomotion', 'Locomotion_type', 'Grasping', 'Grasp_type', 'Group']

Target distribution:
Group
ASD        2093
Non-ASD    1065
Name: count, dtype: int64


In [19]:
df_clean['Location'] = df_clean['Location'].replace('unknown', 'none')
print("Replaced 'unknown' with 'none' in Location")
print("\nLocation values now:")
print(df_clean['Location'].value_counts().sort_index())

Replaced 'unknown' with 'none' in Location

Location values now:
Location
inside private     2277
inside public       253
none                 75
outside private     232
outside public      321
Name: count, dtype: int64


In [20]:
output_path = preprocessing_dir / 'cleaned_data.csv'
df_clean.to_csv(output_path, index=False)
print(f"\nUpdated cleaned data saved to: {output_path}")


Updated cleaned data saved to: /home/aparnabg/orcd/scratch/ml_models/data_preprocessing/cleaned_data.csv


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

root_dir = Path('/home/aparnabg/orcd/scratch/ml_models')
preprocessing_dir = root_dir / 'data_preprocessing'

In [3]:
df = pd.read_csv(preprocessing_dir / 'cleaned_data.csv')
print(f"Loaded: {df.shape}")
X = df.drop('Group', axis=1)
y = df['Group']

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

X_encoded = X.copy()
le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col].astype(str))
    le_dict[col] = le

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print(f"Encoded: {X_encoded.shape}")
print(f"Target: {le_target.classes_}")

Loaded: (3158, 23)
Encoded: (3158, 22)
Target: ['ASD' 'Non-ASD']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (2526, 22), Test: (632, 22)


In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled data created")

Scaled data created


In [6]:
models_no_scaling = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Naive Bayes': GaussianNB()
}

models_with_scaling = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(kernel='rbf', random_state=42)
}

results = {}

print("\nTraining models without scaling:")
for name, model in models_no_scaling.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    
    print(f"\n{name}:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1: {f1:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

print("\nTraining models with scaling:")
for name, model in models_with_scaling.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    
    print(f"\n{name}:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1: {f1:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Training models without scaling:

Random Forest:
Accuracy: 0.6725
Precision: 0.5238
Recall: 0.3099
F1: 0.3894
Confusion Matrix:
[[359  60]
 [147  66]]

Gradient Boosting:
Accuracy: 0.6835
Precision: 0.5783
Recall: 0.2254
F1: 0.3243
Confusion Matrix:
[[384  35]
 [165  48]]

Decision Tree:
Accuracy: 0.6108
Precision: 0.4279
Recall: 0.4601
F1: 0.4434
Confusion Matrix:
[[288 131]
 [115  98]]

Naive Bayes:
Accuracy: 0.5965
Precision: 0.4019
Recall: 0.4038
F1: 0.4028
Confusion Matrix:
[[291 128]
 [127  86]]

Training models with scaling:

Logistic Regression:
Accuracy: 0.6677
Precision: 0.5429
Recall: 0.0892
F1: 0.1532
Confusion Matrix:
[[403  16]
 [194  19]]

SVM:
Accuracy: 0.6756
Precision: 0.6000
Recall: 0.1127
F1: 0.1897
Confusion Matrix:
[[403  16]
 [189  24]]


In [7]:
results_df = pd.DataFrame(results).T.sort_values('accuracy', ascending=False)
print("\nAll Models Comparison:")
print(results_df)


All Models Comparison:
                     accuracy  precision    recall        f1
Gradient Boosting    0.683544   0.578313  0.225352  0.324324
SVM                  0.675633   0.600000  0.112676  0.189723
Random Forest        0.672468   0.523810  0.309859  0.389381
Logistic Regression  0.667722   0.542857  0.089202  0.153226
Decision Tree        0.610759   0.427948  0.460094  0.443439
Naive Bayes          0.596519   0.401869  0.403756  0.402810


In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n5-Fold Cross Validation:")
print("\nModels without scaling:")
for name, model in models_no_scaling.items():
    scores = cross_val_score(model, X_encoded, y_encoded, cv=cv, scoring='accuracy')
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

print("\nModels with scaling:")
for name, model in models_with_scaling.items():
    X_scaled_full = scaler.fit_transform(X_encoded)
    scores = cross_val_score(model, X_scaled_full, y_encoded, cv=cv, scoring='accuracy')
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")


5-Fold Cross Validation:

Models without scaling:
Random Forest: 0.6808 (+/- 0.0138)
Gradient Boosting: 0.6843 (+/- 0.0125)
Decision Tree: 0.6374 (+/- 0.0172)
Naive Bayes: 0.5963 (+/- 0.0253)

Models with scaling:
Logistic Regression: 0.6694 (+/- 0.0057)
SVM: 0.6748 (+/- 0.0113)


In [9]:
best_model_name = results_df.index[0]
print(f"\nBest model: {best_model_name}")
print(f"Accuracy: {results_df.loc[best_model_name, 'accuracy']:.4f}")

if best_model_name in models_no_scaling:
    best_model = models_no_scaling[best_model_name]
    best_model.fit(X_train, y_train)
    y_pred_best = best_model.predict(X_test)
else:
    best_model = models_with_scaling[best_model_name]
    best_model.fit(X_train_scaled, y_train)
    y_pred_best = best_model.predict(X_test_scaled)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=le_target.classes_))


Best model: Gradient Boosting
Accuracy: 0.6835

Detailed Classification Report:
              precision    recall  f1-score   support

         ASD       0.70      0.92      0.79       419
     Non-ASD       0.58      0.23      0.32       213

    accuracy                           0.68       632
   macro avg       0.64      0.57      0.56       632
weighted avg       0.66      0.68      0.64       632



In [10]:
if hasattr(best_model, 'feature_importances_'):
    feature_imp = pd.DataFrame({
        'feature': X_encoded.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 Feature Importances:")
    print(feature_imp.head(10))


Top 10 Feature Importances:
                 feature  importance
0                    Age    0.290182
5             #_children    0.149393
4               #_adults    0.076751
19       Locomotion_type    0.073839
8   #_people_interacting    0.048496
11          Support_type    0.048131
2                Context    0.041405
16              RMM_type    0.036480
13          Gesture_type    0.035704
14         Vocalizations    0.035219


In [11]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import matplotlib.pyplot as plt

print("Dimensionality Reduction Techniques Lisg")
print("\n1. PCA - Principal Component Analysis")
print("2. LDA - Linear Discriminant Analysis")
print("3. Feature Selection - Select K Best")
print("4. t-SNE - for visualization")

Dimensionality Reduction Techniques to Try:

1. PCA - Principal Component Analysis
2. LDA - Linear Discriminant Analysis
3. Feature Selection - Select K Best
4. t-SNE - for visualization


In [12]:
print("\nCurrent feature count:", X_encoded.shape[1])
print("\nTrying PCA to reduce dimensions:")

pca = PCA(random_state=42)
pca.fit(X_train_scaled)

cumsum = np.cumsum(pca.explained_variance_ratio_)
print("\nVariance explained by components:")
for i in range(min(10, len(cumsum))):
    print(f"PC{i+1}: {pca.explained_variance_ratio_[i]:.4f} (cumsum: {cumsum[i]:.4f})")

n_components_95 = np.argmax(cumsum >= 0.95) + 1
print(f"\nComponents needed for 95% variance: {n_components_95}")


Current feature count: 22

Trying PCA to reduce dimensions:

Variance explained by components:
PC1: 0.1177 (cumsum: 0.1177)
PC2: 0.0925 (cumsum: 0.2102)
PC3: 0.0836 (cumsum: 0.2938)
PC4: 0.0748 (cumsum: 0.3687)
PC5: 0.0691 (cumsum: 0.4377)
PC6: 0.0625 (cumsum: 0.5002)
PC7: 0.0572 (cumsum: 0.5575)
PC8: 0.0510 (cumsum: 0.6084)
PC9: 0.0458 (cumsum: 0.6542)
PC10: 0.0423 (cumsum: 0.6965)

Components needed for 95% variance: 19


In [13]:
n_components_list = [5, 10, 15, 20]

print("\nTesting different PCA components:")
for n in n_components_list:
    pca = PCA(n_components=n, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    print(f"\n{n} components (variance: {pca.explained_variance_ratio_.sum():.4f}):")
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train_pca, y_train)
    y_pred = rf.predict(X_test_pca)
    print(f"  Random Forest: {accuracy_score(y_test, y_pred):.4f}")
    
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_train_pca, y_train)
    y_pred = lr.predict(X_test_pca)
    print(f"  Logistic Regression: {accuracy_score(y_test, y_pred):.4f}")
    
    svm = SVC(kernel='rbf', random_state=42)
    svm.fit(X_train_pca, y_train)
    y_pred = svm.predict(X_test_pca)
    print(f"  SVM: {accuracy_score(y_test, y_pred):.4f}")


Testing different PCA components:

5 components (variance: 0.4377):
  Random Forest: 0.6598
  Logistic Regression: 0.6630
  SVM: 0.6630

10 components (variance: 0.6965):
  Random Forest: 0.6566
  Logistic Regression: 0.6646
  SVM: 0.6741

15 components (variance: 0.8709):
  Random Forest: 0.6677
  Logistic Regression: 0.6630
  SVM: 0.6788

20 components (variance: 0.9798):
  Random Forest: 0.6693
  Logistic Regression: 0.6709
  SVM: 0.6772


In [15]:
print("\nTrying LDA - Linear Discriminant Analysis:")
lda = LinearDiscriminantAnalysis()
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)

print(f"LDA components: {X_train_lda.shape[1]}")

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_lda, y_train)
y_pred = rf.predict(X_test_lda)
print(f"Random Forest: {accuracy_score(y_test, y_pred):.4f}")

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_lda, y_train)
y_pred = lr.predict(X_test_lda)
print(f"Logistic Regression: {accuracy_score(y_test, y_pred):.4f}")

svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_lda, y_train)
y_pred = svm.predict(X_test_lda)
print(f"SVM: {accuracy_score(y_test, y_pred):.4f}")


Trying LDA - Linear Discriminant Analysis:
LDA components: 1
Random Forest: 0.5759
Logistic Regression: 0.6693
SVM: 0.6741


In [16]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

print("\nFeature Selection with SelectKBest:")

for k in [5, 10, 15]:
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_test_selected = selector.transform(X_test_scaled)
    
    print(f"\nTop {k} features:")
    selected_features = X_encoded.columns[selector.get_support()].tolist()
    print(selected_features)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train_selected, y_train)
    y_pred = rf.predict(X_test_selected)
    print(f"Random Forest: {accuracy_score(y_test, y_pred):.4f}")
    
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_train_selected, y_train)
    y_pred = lr.predict(X_test_selected)
    print(f"Logistic Regression: {accuracy_score(y_test, y_pred):.4f}")


Feature Selection with SelectKBest:

Top 5 features:
['#_adults', 'Support_type', 'Vocalizations', 'Locomotion', 'Locomotion_type']
Random Forest: 0.6709
Logistic Regression: 0.6630

Top 10 features:
['#_adults', '#_people_interacting', 'Child_constrained', 'Support_type', 'Gesture_type', 'Vocalizations', 'RMM_type', 'Response_to_name', 'Locomotion', 'Locomotion_type']
Random Forest: 0.6250
Logistic Regression: 0.6741

Top 15 features:
['#_adults', '#_children', 'Interaction_with_child', '#_people_interacting', 'Child_constrained', 'Support_type', 'Gestures', 'Gesture_type', 'Vocalizations', 'RMM', 'RMM_type', 'Response_to_name', 'Locomotion', 'Locomotion_type', 'Grasping']
Random Forest: 0.6377
Logistic Regression: 0.6661


In [17]:
from sklearn.feature_selection import RFE

print("\nRecursive Feature Elimination (RFE):")

for n_features in [10, 15]:
    rfe = RFE(estimator=LogisticRegression(random_state=42, max_iter=1000), n_features_to_select=n_features)
    X_train_rfe = rfe.fit_transform(X_train_scaled, y_train)
    X_test_rfe = rfe.transform(X_test_scaled)
    
    print(f"\nRFE {n_features} features:")
    selected_features = X_encoded.columns[rfe.support_].tolist()
    print(selected_features)
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train_rfe, y_train)
    y_pred = rf.predict(X_test_rfe)
    print(f"Random Forest: {accuracy_score(y_test, y_pred):.4f}")
    
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_train_rfe, y_train)
    y_pred = lr.predict(X_test_rfe)
    print(f"Logistic Regression: {accuracy_score(y_test, y_pred):.4f}")


Recursive Feature Elimination (RFE):

RFE 10 features:
['#_adults', '#_children', 'Interaction_with_child', 'Support_type', 'Gesture_type', 'Vocalizations', 'RMM_type', 'Response_to_name', 'Locomotion', 'Locomotion_type']
Random Forest: 0.6345
Logistic Regression: 0.6677

RFE 15 features:
['Age', '#_adults', '#_children', 'Interaction_with_child', 'Child_constrained', 'Supports', 'Support_type', 'Gesture_type', 'Vocalizations', 'RMM', 'RMM_type', 'Response_to_name', 'Locomotion', 'Locomotion_type', 'Grasping']
Random Forest: 0.6630
Logistic Regression: 0.6677


In [8]:

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
print("\nOriginal class distribution:")
print(f"Train: {pd.Series(y_train).value_counts().to_dict()}")

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE: {pd.Series(y_train_smote).value_counts().to_dict()}")

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_smote, y_train_smote)
y_pred = rf.predict(X_test_scaled)
print(f"\nRandom Forest with SMOTE: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_smote, y_train_smote)
y_pred = lr.predict(X_test_scaled)
print(f"\nLogistic Regression with SMOTE: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))


# Gradient Boosting + SMOTE
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_smote, y_train_smote)
y_pred = gb.predict(X_test_scaled)
print("\nGradient Boosting + SMOTE:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

# XGBoost + SMOTE
xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb.fit(X_train_smote, y_train_smote)
y_pred = xgb.predict(X_test_scaled)
print("\nXGBoost + SMOTE:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

# Voting Classifier + SMOTE
voting = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ('lr', LogisticRegression(random_state=42, max_iter=1000))
    ],
    voting='hard'
)
voting.fit(X_train_smote, y_train_smote)
y_pred = voting.predict(X_test_scaled)
print("\nVoting Classifier + SMOTE:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))


Original class distribution:
Train: {0: 1674, 1: 852}
After SMOTE: {1: 1674, 0: 1674}

Random Forest with SMOTE: 0.6661
              precision    recall  f1-score   support

         ASD       0.73      0.80      0.76       419
     Non-ASD       0.51      0.40      0.45       213

    accuracy                           0.67       632
   macro avg       0.62      0.60      0.60       632
weighted avg       0.65      0.67      0.66       632


Logistic Regression with SMOTE: 0.5396
              precision    recall  f1-score   support

         ASD       0.71      0.51      0.60       419
     Non-ASD       0.38      0.59      0.46       213

    accuracy                           0.54       632
   macro avg       0.55      0.55      0.53       632
weighted avg       0.60      0.54      0.55       632


Gradient Boosting + SMOTE:
Accuracy: 0.6535
              precision    recall  f1-score   support

         ASD       0.73      0.76      0.74       419
     Non-ASD       0.48      0.

In [19]:
from sklearn.ensemble import VotingClassifier

print("\nVoting Classifier - Ensemble of best models:")

rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(random_state=42, max_iter=1000)

voting_hard = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('lr', lr)],
    voting='hard'
)

voting_hard.fit(X_train_scaled, y_train)
y_pred = voting_hard.predict(X_test_scaled)

print(f"Hard Voting: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))


Voting Classifier - Ensemble of best models:
Hard Voting: 0.6851
              precision    recall  f1-score   support

         ASD       0.69      0.94      0.80       419
     Non-ASD       0.61      0.18      0.28       213

    accuracy                           0.69       632
   macro avg       0.65      0.56      0.54       632
weighted avg       0.67      0.69      0.62       632



In [22]:

from xgboost import XGBClassifier

print("\nTrying XGBoost:")
xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb.fit(X_train_scaled, y_train)
y_pred = xgb.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

xgb_balanced = XGBClassifier(
    n_estimators=100, 
    random_state=42, 
    eval_metric='logloss',
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1])
)
xgb_balanced.fit(X_train_scaled, y_train)
y_pred = xgb_balanced.predict(X_test_scaled)

print(f"\nXGBoost with balanced weights: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))



Trying XGBoost:
Accuracy: 0.6566
              precision    recall  f1-score   support

         ASD       0.71      0.82      0.76       419
     Non-ASD       0.49      0.33      0.39       213

    accuracy                           0.66       632
   macro avg       0.60      0.58      0.58       632
weighted avg       0.63      0.66      0.64       632


XGBoost with balanced weights: 0.6614
              precision    recall  f1-score   support

         ASD       0.75      0.74      0.74       419
     Non-ASD       0.50      0.51      0.50       213

    accuracy                           0.66       632
   macro avg       0.62      0.62      0.62       632
weighted avg       0.66      0.66      0.66       632



In [10]:
from sklearn.decomposition import PCA

n_components_list = [2, 5, 10, 15, 20]

print("Testing remaining models with PCA:")

for n in n_components_list:
    pca = PCA(n_components=n, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    print(f"\n{n} components (variance: {pca.explained_variance_ratio_.sum():.4f}):")
    
    # Gradient Boosting
    gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
    gb.fit(X_train_pca, y_train)
    y_pred = gb.predict(X_test_pca)
    print(f"  Gradient Boosting: {accuracy_score(y_test, y_pred):.4f}")
    
    # XGBoost
    xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
    xgb.fit(X_train_pca, y_train)
    y_pred = xgb.predict(X_test_pca)
    print(f"  XGBoost: {accuracy_score(y_test, y_pred):.4f}")
    
    # Decision Tree
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_train_pca, y_train)
    y_pred = dt.predict(X_test_pca)
    print(f"  Decision Tree: {accuracy_score(y_test, y_pred):.4f}")
    
    # Naive Bayes
    nb = GaussianNB()
    nb.fit(X_train_pca, y_train)
    y_pred = nb.predict(X_test_pca)
    print(f"  Naive Bayes: {accuracy_score(y_test, y_pred):.4f}")

Testing remaining models with PCA:

2 components (variance: 0.2102):
  Gradient Boosting: 0.6551
  XGBoost: 0.6345
  Decision Tree: 0.5997
  Naive Bayes: 0.6630

5 components (variance: 0.4377):
  Gradient Boosting: 0.6677
  XGBoost: 0.6155
  Decision Tree: 0.5807
  Naive Bayes: 0.6630

10 components (variance: 0.6965):
  Gradient Boosting: 0.6741
  XGBoost: 0.6282
  Decision Tree: 0.5839
  Naive Bayes: 0.6677

15 components (variance: 0.8709):
  Gradient Boosting: 0.6820
  XGBoost: 0.6503
  Decision Tree: 0.5870
  Naive Bayes: 0.6535

20 components (variance: 0.9798):
  Gradient Boosting: 0.6598
  XGBoost: 0.6329
  Decision Tree: 0.5554
  Naive Bayes: 0.6234
